# Question 3: Image Completion Generator (Step 2)

In the last notebook with Edge Connect Generator, predicting edges first provides structural guidance for the image completion model, helping preserve object boundaries, facial geometry, and texture continuity in occluded regions.

This notebook trains the image completion generator for the edge-guided pipeline. It uses the masked image, predicted edge map, and binary mask as input to reconstruct the missing dog image content. This model is stronger than the plain U-Net baseline because it receives explicit structural guidance from the edge generator in step 1.


Notebook outline:
* Setup and paths
* Load train / val / test split CSVs
* Create random masks
* Generate edge maps
* Build EdgeInpaintingDataset
  * Stage 1: Edge Generator
  * **Stage 2: Completion Generator**
* Patch Discriminator
* Train mini EdgeConnect GAN
* Evaluate against baseline UNet
* Visualize results
* Save model, history, metrics, and examples

https://www.kaggle.com/datasets/jessicali9530/stanford-dogs-dataset


# Environment Set Up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

## Load Data

In [ ]:
from pathlib import Path

# Main project folder
BASE_DIR = Path("/content/drive/MyDrive/Adv CV/CV Project Folder")

# Dataset folders
ANNOTATION_DIR = BASE_DIR / "Annotation"
IMAGE_DIR = BASE_DIR / "Images"

print("Annotation exists:", ANNOTATION_DIR.exists())
print("Image folder exists:", IMAGE_DIR.exists())

In [ ]:
print("Annotation contents:")
print(list(ANNOTATION_DIR.iterdir())[:10])

print("Image contents:")
print(list(IMAGE_DIR.iterdir())[:10])

In [ ]:
# define image paths
image_paths = sorted(list(IMAGE_DIR.rglob("*.jpg")))

print("Number of images:", len(image_paths))
print("Example image path:", image_paths[0])

In [ ]:
# sanity check
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(image_paths[0]).convert("RGB")

plt.imshow(img)
plt.axis("off")
plt.show()

## Model Preparation

For EdgeConnect-based GAN approach, we need the following:

* Input: masked image, masked edges, mask

* Output: completed edge map



In [ ]:
import torch
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

### Image transform

In [ ]:
# Faster training setting
# Use 128 for quick GAN/EdgeConnect-style experiments.
# You can switch back to 256 later for final-quality results if needed.
IMG_SIZE = 128

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),   # range: [0, 1]
])

### Define a random **mask** helper

We do this because for inpainting, mask convention matters where:
* 1 = visible/known area
* 0 = missing/hole area

In [ ]:
def create_random_rect_mask(image, min_size=50, max_size=120):
    _, h, w = image.shape

    mask = torch.ones((1, h, w))

    box_w = random.randint(min_size, max_size)
    box_h = random.randint(min_size, max_size)

    x1 = random.randint(0, w - box_w)
    y1 = random.randint(0, h - box_h)

    mask[:, y1:y1+box_h, x1:x1+box_w] = 0

    return mask

### Define a simple **edge** helper

This will return an edge map from the clean dog image and EdgeConnect uses edges as structural guidance.

In [ ]:
import cv2

def create_edge_map(image_tensor):
    """
    image_tensor: torch tensor, shape [3, H, W], range [0, 1]
    returns: torch tensor, shape [1, H, W], range [0, 1]
    """
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    image_np = (image_np * 255).astype(np.uint8)

    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, threshold1=100, threshold2=200)

    edges = edges.astype(np.float32) / 255.0
    edges = torch.from_numpy(edges).unsqueeze(0)

    return edges

### Prepare dataset class

In [ ]:
class DogEdgeInpaintingDataset(Dataset):
    """
    Efficient version:
    - Caches the clean image edge map after it is computed once.
    - Still creates a new random mask each time, so training keeps data augmentation.
    """
    def __init__(self, image_paths, transform=None, mask_fn=None, cache_edges=True):
        self.image_paths = image_paths
        self.transform = transform
        self.mask_fn = mask_fn
        self.cache_edges = cache_edges
        self.edge_cache = {}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        mask = self.mask_fn(image)

        # Important speedup: Canny edge detection is CPU-heavy.
        # Cache the full clean edge map once per image instead of recomputing it every epoch.
        if self.cache_edges and idx in self.edge_cache:
            edge = self.edge_cache[idx]
        else:
            edge = create_edge_map(image)
            if self.cache_edges:
                self.edge_cache[idx] = edge

        masked_image = image * mask
        masked_edge = edge * mask

        # Stage 1 edge generator input: masked image + masked edge + mask
        edge_input = torch.cat([masked_image, masked_edge, mask], dim=0)

        # Stage 2 image generator input: masked image + full edge + mask
        image_input = torch.cat([masked_image, edge, mask], dim=0)

        return {
            "image": image,
            "mask": mask,
            "masked_image": masked_image,
            "edge": edge,
            "masked_edge": masked_edge,
            "edge_input": edge_input,
            "image_input": image_input,
        }

### DataLoader sanity check

In [ ]:
# Initial sanity-check dataset using rectangle masks
dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_random_rect_mask,
    cache_edges=True
)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Faster DataLoader settings
BATCH_SIZE = 8 if torch.cuda.is_available() else 4
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

print("Batch size:", BATCH_SIZE)
print("Train batches per epoch:", len(train_loader))

Sanity check before training model:

In [ ]:
sample = dataset[0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

As shown, the mask here might be a bit too simple.

Optional: I can also increase batch size

In [ ]:
#BATCH_SIZE = 8

#train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
#val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
#test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### Improve the mask generator

To help EdgeConnect perform better, we can use irregular masks, brush strokes, random holes or larger missing regions.

In [ ]:
import cv2
import numpy as np
import random
import torch

def create_irregular_mask(image, max_strokes=4):
    _, h, w = image.shape
    mask = np.ones((h, w), np.float32)

    for _ in range(random.randint(1, max_strokes)):
        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
        x2, y2 = random.randint(0, w-1), random.randint(0, h-1)

        thickness = random.randint(6, 25)
        cv2.line(mask, (x1, y1), (x2, y2), 0, thickness)

        radius = random.randint(6, 18)
        cv2.circle(mask, (x2, y2), radius, 0, -1)

    return torch.from_numpy(mask).unsqueeze(0)

In [ ]:
# can make the mask less chaotic/random
def create_mixed_mask(image):
    if random.random() < 0.5:
        return create_random_rect_mask(image, min_size=40, max_size=100)
    else:
        return create_irregular_mask(image, max_strokes=3)

In [ ]:
# update the dataset
dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_irregular_mask
)

Sanity check again:

In [ ]:
sample = dataset[1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

I observed that some masks would fall into the background (not on the dog). Therefore I added the following setup:

In [ ]:
def mask_touches_foreground(mask, image, threshold=0.08):
    """
    Rough heuristic: checks whether the missing area overlaps with non-background-ish regions.
    Not perfect, but useful when you don't have dog segmentation masks.
    """
    missing = (mask == 0).float()
    return missing.mean().item() > threshold

### Generate masks around the image center
(Because dogs are usually near the center)

In [ ]:
def create_center_biased_irregular_mask(image, max_strokes=3):
    _, h, w = image.shape
    mask = np.ones((h, w), np.float32)

    for _ in range(random.randint(1, max_strokes)):
        cx = random.randint(w // 4, 3 * w // 4)
        cy = random.randint(h // 4, 3 * h // 4)

        x2 = min(max(cx + random.randint(-60, 60), 0), w - 1)
        y2 = min(max(cy + random.randint(-60, 60), 0), h - 1)

        thickness = random.randint(6, 22)
        cv2.line(mask, (cx, cy), (x2, y2), 0, thickness)

        radius = random.randint(6, 18)
        cv2.circle(mask, (x2, y2), radius, 0, -1)

    return torch.from_numpy(mask).unsqueeze(0)

In [ ]:
def create_mixed_mask(image):
    r = random.random()

    if r < 0.35:
        return create_random_rect_mask(image, min_size=40, max_size=100)
    elif r < 0.80:
        return create_center_biased_irregular_mask(image, max_strokes=3)
    else:
        return create_irregular_mask(image, max_strokes=3)

Sanity check:

In [ ]:
# Final efficient dataset + dataloaders using the mixed/center-biased mask generator

dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_mixed_mask,
    cache_edges=True
)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 8 if torch.cuda.is_available() else 4
NUM_WORKERS = 2

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print("Dataset size:", len(dataset))
print("Train / Val / Test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("Train batches per epoch:", len(train_loader))

In [ ]:
sample = dataset[0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(sample["image"].permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(sample["mask"].squeeze(), cmap="gray")
axes[1].set_title("Mask")

axes[2].imshow(sample["masked_image"].permute(1, 2, 0))
axes[2].set_title("Masked Image")

axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
axes[3].set_title("Full Edge")

axes[4].imshow(sample["masked_edge"].squeeze(), cmap="gray")
axes[4].set_title("Masked Edge")

for ax in axes:
    ax.axis("off")

plt.show()

This is slightly better, so we can proceed with this mask generator.

## Train a simple U-Net Style Edge generator

Current setup:
* edge_input  # [masked image, masked edge, mask], 5 channels
* edge        # target full edge map, 1 channel

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EdgeGeneratorUNet(nn.Module):
    """
    Lighter U-Net for faster training.
    base_channels=32 is much faster than the original 64-channel version.
    If you want better final quality later, try base_channels=48 or 64.
    """
    def __init__(self, in_channels=5, out_channels=1, base_channels=32):
        super().__init__()

        c = base_channels
        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(c * 4, c * 8)

        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.final = nn.Conv2d(c, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Return logits. BCEWithLogitsLoss will apply sigmoid internally
        return self.final(d1)

### Define model & Test model shape

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Lighter model = faster epochs
edge_generator = EdgeGeneratorUNet(in_channels=5, out_channels=1, base_channels=32).to(device)

batch = next(iter(train_loader))
edge_input = batch["edge_input"].to(device, non_blocking=True)

with torch.no_grad():
    pred_edge = edge_generator(edge_input)

print("Input shape:", edge_input.shape)
print("Predicted edge shape:", pred_edge.shape)

### Define loss & optimizer

In [ ]:
import torch.nn as nn
import torch.optim as optim

bce_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()

optimizer = optim.Adam(edge_generator.parameters(), lr=1e-4)

# Mixed precision speeds up CUDA training and saves memory.
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

### Train edge generator

In [ ]:
def train_edge_generator(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=5,
    max_train_batches=None,
    max_val_batches=50,
    val_interval=1,
):
    """
    Faster training loop:
    - mixed precision on GPU
    - non_blocking GPU transfer
    - optional max_train_batches for quick experiments
    - optional max_val_batches so validation does not take forever
    """
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        train_steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - train")
        for step, batch in enumerate(pbar):
            if max_train_batches is not None and step >= max_train_batches:
                break

            edge_input = batch["edge_input"].to(device, non_blocking=True)
            target_edge = batch["edge"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                pred_edge = model(edge_input)

                # focus more on missing region
                hole = 1 - mask

                loss_bce = bce_loss(pred_edge, target_edge)
                loss_hole = l1_loss(pred_edge * hole, target_edge * hole)

                loss = loss_bce + 5 * loss_hole

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()
            train_steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / max(train_steps, 1)
        train_losses.append(avg_train_loss)

        # Validate less expensively
        if (epoch + 1) % val_interval == 0:
            model.eval()
            total_val_loss = 0
            val_steps = 0

            with torch.no_grad():
                for step, batch in enumerate(tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - val")):
                    if max_val_batches is not None and step >= max_val_batches:
                        break

                    edge_input = batch["edge_input"].to(device, non_blocking=True)
                    target_edge = batch["edge"].to(device, non_blocking=True)
                    mask = batch["mask"].to(device, non_blocking=True)

                    with torch.amp.autocast("cuda", enabled=use_amp):
                        pred_edge = model(edge_input)
                        hole = 1 - mask

                        loss_bce = bce_loss(pred_edge, target_edge)
                        loss_hole = l1_loss(pred_edge * hole, target_edge * hole)

                        loss = loss_bce + 5 * loss_hole

                    total_val_loss += loss.item()
                    val_steps += 1

            avg_val_loss = total_val_loss / max(val_steps, 1)
        else:
            avg_val_loss = None

        val_losses.append(avg_val_loss)

        if avg_val_loss is None:
            print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}")
        else:
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    return train_losses, val_losses

### Run training

In [ ]:
# Efficient run-training setting
# Start with fewer epochs and a capped number of batches so you can verify the model quickly.
# After it works, remove max_train_batches or increase epochs.

edge_train_losses, edge_val_losses = train_edge_generator(
    edge_generator,
    train_loader,
    val_loader,
    optimizer,
    epochs=10,
    max_train_batches=300,   # set to None for full training
    max_val_batches=50,
    val_interval=1
)

In [ ]:
from pathlib import Path
import torch
import json

# Find output folder
BASE_DIR = Path("/content/drive/MyDrive/Adv CV/CV Project Folder")
OUTPUT_DIR = BASE_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save model weights
torch.save(
    edge_generator.state_dict(),
    OUTPUT_DIR / "edge_generator.pt"
)

# Save training history
history = {
    "train_losses": edge_train_losses,
    "val_losses": edge_val_losses
}

with open(BASE_DIR /"histories" / "edge_training_history.json", "w") as f:
    json.dump(history, f)

print("Saved model + training history.")

In [ ]:
# load edge generation model
edge_generator = EdgeGeneratorUNet(
    in_channels=5,
    out_channels=1,
    base_channels=32
).to(device)

edge_generator.load_state_dict(
    torch.load(OUTPUT_DIR / "edge_generator.pt", map_location=device)
)

edge_generator.eval()

### Visualize Predicted Edges

In [ ]:
def show_edge_prediction(model, dataset, idx=0):
    model.eval()
    sample = dataset[idx]

    edge_input = sample["edge_input"].unsqueeze(0).to(device)

    with torch.no_grad():
        pred_edge = model(edge_input).cpu().squeeze()

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))

    axes[0].imshow(sample["masked_image"].permute(1, 2, 0))
    axes[0].set_title("Masked Image")

    axes[1].imshow(sample["masked_edge"].squeeze(), cmap="gray")
    axes[1].set_title("Masked Edge")

    axes[2].imshow(pred_edge, cmap="gray")
    axes[2].set_title("Predicted Edge")

    axes[3].imshow(sample["edge"].squeeze(), cmap="gray")
    axes[3].set_title("Target Edge")

    for ax in axes:
        ax.axis("off")

    plt.show()

In [ ]:
show_edge_prediction(edge_generator, val_dataset, idx=0)

### Plot Training Curve

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

save_dir_1 = Path("/content/drive/MyDrive/Adv CV/CV Project Folder/histories")
save_dir_1.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(6,4))
plt.plot(edge_train_losses, label="Train Loss")
plt.plot(edge_val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Edge Generator Training Curve")
plt.legend()

plt.savefig(save_dir_1 / "edge_training_curve.png", dpi=300)
plt.show()

# Stage 2: Edge-Guided Image Completion

This section trains the actual image completion model. The model takes the masked dog image, a completed edge map, and the binary mask as input, then predicts the restored RGB image.

* Edge generator = binary prediction → use logits + BCEWithLogitsLoss
* Completion generator = RGB regression → use sigmoid/tanh + L1/L2/perceptual loss

Pipeline:

```text
masked image + predicted/target edge + mask → completion generator → restored dog image
```

For speed and stability, this version uses a lightweight U-Net completion generator with reconstruction losses.


In [ ]:
# Make sure output folders exist
BASE_DIR = Path("/content/drive/MyDrive/Adv CV/CV Project Folder")

MODEL_DIR = BASE_DIR / "models"
HISTORY_DIR = BASE_DIR / "histories"
RESULT_DIR = BASE_DIR / "results"
METRIC_DIR = BASE_DIR / "metrics"

for d in [MODEL_DIR, HISTORY_DIR, RESULT_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", BASE_DIR)

In [ ]:
class EdgeGuidedCompletionUNet(nn.Module):
    """
    Lightweight U-Net for image completion.

    Input channels = 5:
      - masked RGB image: 3 channels
      - completed edge map: 1 channel
      - mask: 1 channel

    Output channels = 3:
      - restored RGB image
    """
    def __init__(self, in_channels=5, out_channels=3, base_channels=32):
        super().__init__()
        c = base_channels

        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(c * 4, c * 8)

        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.final = nn.Conv2d(c, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Images are normalized to [0,1]
        # Sigmoid constrains outputs to valid image range.
        return torch.sigmoid(self.final(d1))

In [ ]:
def get_completed_edge(edge_generator, edge_input):
    """
    Runs the trained edge generator and returns an edge probability map in [0, 1].
    This works whether your edge generator outputs probabilities or logits.
    """
    pred = edge_generator(edge_input)

    # If the model outputs logits, values may go outside [0, 1]. Convert them.
    # If the model already has sigmoid, leave it unchanged.
    if pred.min().item() < 0 or pred.max().item() > 1:
        pred = torch.sigmoid(pred)

    return pred.clamp(0, 1)


def build_completion_input(masked_image, completed_edge, mask):
    """
    Concatenate masked RGB image, completed edge, and mask.
    Shape: [B, 3, H, W] + [B, 1, H, W] + [B, 1, H, W] = [B, 5, H, W]
    """
    return torch.cat([masked_image, completed_edge, mask], dim=1)

In [ ]:
# Initialize completion generator
completion_generator = EdgeGuidedCompletionUNet(
    in_channels=5,
    out_channels=3,
    base_channels=32
).to(device)

completion_optimizer = torch.optim.Adam(
    completion_generator.parameters(),
    lr=1e-4
)

l1_loss = nn.L1Loss()
mse_loss = nn.MSELoss()

use_amp = torch.cuda.is_available()
completion_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Completion generator initialized on:", device)

In [ ]:
def train_completion_generator(
    completion_model,
    edge_model,
    train_loader,
    val_loader,
    optimizer,
    epochs=15,
    max_train_batches=300,
    max_val_batches=50,
    use_predicted_edges=True,
):
    """
    Trains the image completion model.

    use_predicted_edges=True:
        Uses the trained edge generator output. This matches the real pipeline.

    use_predicted_edges=False:
        Uses ground-truth target edges. This is faster/easier and gives an upper-bound sanity check.
    """
    train_losses = []
    val_losses = []

    edge_model.eval()

    for epoch in range(epochs):
        completion_model.train()
        total_train_loss = 0
        train_steps = 0

        pbar = tqdm(train_loader, desc=f"Completion Epoch {epoch+1}/{epochs} - train")
        for step, batch in enumerate(pbar):
            if max_train_batches is not None and step >= max_train_batches:
                break

            image = batch["image"].to(device, non_blocking=True)
            masked_image = batch["masked_image"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)
            edge_input = batch["edge_input"].to(device, non_blocking=True)
            target_edge = batch["edge"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                if use_predicted_edges:
                    completed_edge = get_completed_edge(edge_model, edge_input)
                else:
                    completed_edge = target_edge

            with torch.amp.autocast("cuda", enabled=use_amp):
                completion_input = build_completion_input(masked_image, completed_edge, mask)
                pred_image = completion_model(completion_input)

                # Preserve the known region from the original masked input.
                # This makes the model focus on the missing hole instead of changing everything.
                comp_image = pred_image * (1 - mask) + image * mask

                hole = 1 - mask

                loss_hole = l1_loss(pred_image * hole, image * hole)
                loss_valid = l1_loss(pred_image * mask, image * mask)
                loss_full = l1_loss(comp_image, image)
                loss_mse = mse_loss(comp_image, image)

                # Heavier weight on the missing region, since that is the real task.
                loss = 6.0 * loss_hole + 1.0 * loss_valid + 1.0 * loss_full + 0.5 * loss_mse

            completion_scaler.scale(loss).backward()
            completion_scaler.step(optimizer)
            completion_scaler.update()

            total_train_loss += loss.item()
            train_steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / max(train_steps, 1)
        train_losses.append(avg_train_loss)

        completion_model.eval()
        total_val_loss = 0
        val_steps = 0

        with torch.no_grad():
            for step, batch in enumerate(tqdm(val_loader, desc=f"Completion Epoch {epoch+1}/{epochs} - val")):
                if max_val_batches is not None and step >= max_val_batches:
                    break

                image = batch["image"].to(device, non_blocking=True)
                masked_image = batch["masked_image"].to(device, non_blocking=True)
                mask = batch["mask"].to(device, non_blocking=True)
                edge_input = batch["edge_input"].to(device, non_blocking=True)
                target_edge = batch["edge"].to(device, non_blocking=True)

                if use_predicted_edges:
                    completed_edge = get_completed_edge(edge_model, edge_input)
                else:
                    completed_edge = target_edge

                with torch.amp.autocast("cuda", enabled=use_amp):
                    completion_input = build_completion_input(masked_image, completed_edge, mask)
                    pred_image = completion_model(completion_input)
                    comp_image = pred_image * (1 - mask) + image * mask

                    hole = 1 - mask
                    loss_hole = l1_loss(pred_image * hole, image * hole)
                    loss_valid = l1_loss(pred_image * mask, image * mask)
                    loss_full = l1_loss(comp_image, image)
                    loss_mse = mse_loss(comp_image, image)
                    loss = 6.0 * loss_hole + 1.0 * loss_valid + 1.0 * loss_full + 0.5 * loss_mse

                total_val_loss += loss.item()
                val_steps += 1

        avg_val_loss = total_val_loss / max(val_steps, 1)
        val_losses.append(avg_val_loss)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Completion Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

    return train_losses, val_losses

In [ ]:
# Run image completion training
# Start small first. If this looks stable, increase max_train_batches to 500 later.

completion_train_losses, completion_val_losses = train_completion_generator(
    completion_generator,
    edge_generator,
    train_loader,
    val_loader,
    completion_optimizer,
    epochs=5,
    max_train_batches=200,
    max_val_batches=50,
    use_predicted_edges=True,
)

In [ ]:
# Plot completion training curves
plt.figure(figsize=(6, 4))
plt.plot(completion_train_losses, label="Train Loss")
plt.plot(completion_val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Image Completion Training Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def show_completion_result(completion_model, edge_model, dataset, idx=0, threshold_edge=False, save_path=None):
    completion_model.eval()
    edge_model.eval()

    sample = dataset[idx]

    image = sample["image"].unsqueeze(0).to(device)
    masked_image = sample["masked_image"].unsqueeze(0).to(device)
    mask = sample["mask"].unsqueeze(0).to(device)
    edge_input = sample["edge_input"].unsqueeze(0).to(device)
    target_edge = sample["edge"].unsqueeze(0).to(device)

    with torch.no_grad():
        completed_edge = get_completed_edge(edge_model, edge_input)
        if threshold_edge:
            completed_edge = (completed_edge > 0.5).float()

        completion_input = build_completion_input(masked_image, completed_edge, mask)
        pred_image = completion_model(completion_input)
        comp_image = pred_image * (1 - mask) + image * mask

    image_np = image.cpu().squeeze(0).permute(1, 2, 0).numpy()
    masked_np = masked_image.cpu().squeeze(0).permute(1, 2, 0).numpy()
    pred_edge_np = completed_edge.cpu().squeeze().numpy()
    target_edge_np = target_edge.cpu().squeeze().numpy()
    pred_np = pred_image.cpu().squeeze(0).permute(1, 2, 0).numpy()
    comp_np = comp_image.cpu().squeeze(0).permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 6, figsize=(18, 4))

    axes[0].imshow(masked_np)
    axes[0].set_title("Masked Input")

    axes[1].imshow(pred_edge_np, cmap="gray")
    axes[1].set_title("Predicted Edge")

    axes[2].imshow(target_edge_np, cmap="gray")
    axes[2].set_title("Target Edge")

    axes[3].imshow(pred_np)
    axes[3].set_title("Generated Image")

    axes[4].imshow(comp_np)
    axes[4].set_title("Final Completion")

    axes[5].imshow(image_np)
    axes[5].set_title("Ground Truth")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved figure to:", save_path)

    plt.show()

In [ ]:
# Visualize several validation examples
for i in range(3):
    show_completion_result(
        completion_generator,
        edge_generator,
        val_dataset,
        idx=i,
        threshold_edge=False,
        save_path=RESULT_DIR / f"completion_example_{i}.png"
    )

In [ ]:
def evaluate_completion_model(completion_model, edge_model, loader, max_batches=50, use_predicted_edges=True):
    completion_model.eval()
    edge_model.eval()

    full_l1_values = []
    hole_l1_values = []
    mse_values = []

    with torch.no_grad():
        for step, batch in enumerate(tqdm(loader, desc="Evaluating completion")):
            if max_batches is not None and step >= max_batches:
                break

            image = batch["image"].to(device, non_blocking=True)
            masked_image = batch["masked_image"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)
            edge_input = batch["edge_input"].to(device, non_blocking=True)
            target_edge = batch["edge"].to(device, non_blocking=True)

            if use_predicted_edges:
                completed_edge = get_completed_edge(edge_model, edge_input)
            else:
                completed_edge = target_edge

            completion_input = build_completion_input(masked_image, completed_edge, mask)
            pred_image = completion_model(completion_input)
            comp_image = pred_image * (1 - mask) + image * mask

            hole = 1 - mask

            full_l1 = F.l1_loss(comp_image, image)
            hole_l1 = F.l1_loss(pred_image * hole, image * hole)
            mse = F.mse_loss(comp_image, image)

            full_l1_values.append(full_l1.item())
            hole_l1_values.append(hole_l1.item())
            mse_values.append(mse.item())

    metrics = {
        "full_l1": float(np.mean(full_l1_values)),
        "missing_region_l1": float(np.mean(hole_l1_values)),
        "mse": float(np.mean(mse_values)),
    }

    return metrics

completion_metrics = evaluate_completion_model(
    completion_generator,
    edge_generator,
    test_loader,
    max_batches=50,
    use_predicted_edges=True,
)

#completion_metrics

In [ ]:
# Save completion model, losses, and metrics
torch.save(completion_generator.state_dict(), MODEL_DIR / "completion_generator_edge_guided.pt")

history = {
    "completion_train_losses": completion_train_losses,
    "completion_val_losses": completion_val_losses,
}

with open(HISTORY_DIR / "completion_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

with open(METRIC_DIR / "completion_metrics.json", "w") as f:
    json.dump(completion_metrics, f, indent=2)

print("Saved completion model to:", MODEL_DIR / "completion_generator_edge_guided.pt")
print("Saved history to:", HISTORY_DIR / "completion_training_history.json")
print("Saved metrics to:", METRIC_DIR / "completion_metrics.json")
print("Metrics:", completion_metrics)

## Short catch-up

This stage uses an edge-guided image completion model. After the edge generator predicts missing structural contours, a U-Net-style completion generator takes the masked RGB image, predicted edge map, and binary mask as input. The model is trained with reconstruction losses that emphasize the missing region, producing a restored dog image while preserving the known visible pixels.

However, clearly, U-Net is not enough for realistic dog image completion under large occlusion as shown above. The U-Net model is only able to learn a blurry average fill, not realistic dog structure. Although reconstruction losses decreased consistently during training (refer to the loss curve), qualitative outputs remained blurry under large occlusions. This reflects a known limitation of pixel-wise reconstruction objectives, which often produce overly smooth image completions lacking realistic texture and semantic detail.

## Fine-Tuning: Improving with smaller mask & train longer

* Smaller mask
* Longer training

In [ ]:
# Use smaller masks

def create_smaller_mixed_mask(image):
    r = random.random()

    # 1 = visible, 0 = missing
    if r < 0.45:
        return create_random_rect_mask(image, min_size=25, max_size=70)
    elif r < 0.85:
        return create_center_biased_irregular_mask(image, max_strokes=2)
    else:
        return create_irregular_mask(image, max_strokes=2)


dataset = DogEdgeInpaintingDataset(
    image_paths=image_paths,
    transform=transform,
    mask_fn=create_smaller_mixed_mask,
    cache_edges=True
)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 8 if torch.cuda.is_available() else 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print("Improved dataloaders ready.")
print("Train batches:", len(train_loader))

In [ ]:
# Let the model train longer

completion_generator = EdgeGuidedCompletionUNet(
    in_channels=5,
    out_channels=3,
    base_channels=48
).to(device)

completion_optimizer = torch.optim.AdamW(
    completion_generator.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

completion_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    completion_optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

### Train the fine-tuned image generator

In [ ]:
def train_completion_generator_improved(
    completion_model,
    edge_model,
    train_loader,
    val_loader,
    optimizer,
    scheduler=None,
    epochs=10,
    max_train_batches=500,
    max_val_batches=100,
    use_predicted_edges=True,
):
    train_losses = []
    val_losses = []

    edge_model.eval()

    for epoch in range(epochs):
        completion_model.train()
        total_train_loss = 0
        train_steps = 0

        pbar = tqdm(train_loader, desc=f"Improved Completion Epoch {epoch+1}/{epochs}")

        for step, batch in enumerate(pbar):
            if max_train_batches is not None and step >= max_train_batches:
                break

            image = batch["image"].to(device, non_blocking=True)
            masked_image = batch["masked_image"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)
            edge_input = batch["edge_input"].to(device, non_blocking=True)
            target_edge = batch["edge"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                completed_edge = (
                    get_completed_edge(edge_model, edge_input)
                    if use_predicted_edges
                    else target_edge
                )

            with torch.amp.autocast("cuda", enabled=use_amp):
                completion_input = build_completion_input(masked_image, completed_edge, mask)
                pred_image = completion_model(completion_input)

                hole = 1 - mask
                comp_image = pred_image * hole + image * mask

                loss_hole = F.l1_loss(pred_image * hole, image * hole)
                loss_valid = F.l1_loss(pred_image * mask, image * mask)
                loss_full = F.l1_loss(comp_image, image)
                loss_mse = F.mse_loss(comp_image, image)

                # More focus on missing region, less pressure to reconstruct known region
                loss = 10.0 * loss_hole + 0.5 * loss_valid + 1.0 * loss_full + 0.25 * loss_mse

            completion_scaler.scale(loss).backward()
            completion_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(completion_model.parameters(), max_norm=1.0)
            completion_scaler.step(optimizer)
            completion_scaler.update()

            total_train_loss += loss.item()
            train_steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / max(train_steps, 1)
        train_losses.append(avg_train_loss)

        completion_model.eval()
        total_val_loss = 0
        val_steps = 0

        with torch.no_grad():
            for step, batch in enumerate(val_loader):
                if max_val_batches is not None and step >= max_val_batches:
                    break

                image = batch["image"].to(device, non_blocking=True)
                masked_image = batch["masked_image"].to(device, non_blocking=True)
                mask = batch["mask"].to(device, non_blocking=True)
                edge_input = batch["edge_input"].to(device, non_blocking=True)
                target_edge = batch["edge"].to(device, non_blocking=True)

                completed_edge = (
                    get_completed_edge(edge_model, edge_input)
                    if use_predicted_edges
                    else target_edge
                )

                completion_input = build_completion_input(masked_image, completed_edge, mask)
                pred_image = completion_model(completion_input)

                hole = 1 - mask
                comp_image = pred_image * hole + image * mask

                val_loss = F.l1_loss(comp_image, image)

                total_val_loss += val_loss.item()
                val_steps += 1

        avg_val_loss = total_val_loss / max(val_steps, 1)
        val_losses.append(avg_val_loss)

        if scheduler is not None:
            scheduler.step(avg_val_loss)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

    return train_losses, val_losses

In [ ]:
improved_train_losses, improved_val_losses = train_completion_generator_improved(
    completion_generator,
    edge_generator,
    train_loader,
    val_loader,
    completion_optimizer,
    scheduler=completion_scheduler,
    epochs=10,
    max_train_batches=500,
    max_val_batches=100,
    use_predicted_edges=True,
)

### Plot fine-tuned training curve

In [ ]:
# Plot improved training curve
plt.figure(figsize=(6, 4))
plt.plot(improved_train_losses, label="Improved Train Loss")
plt.plot(improved_val_losses, label="Improved Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Improved Image Completion Training Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Visualize results

In [ ]:
# Visualize improved results
for i in range(5):
    show_completion_result(
        completion_generator,
        edge_generator,
        val_dataset,
        idx=i,
        threshold_edge=False,
        save_path=RESULT_DIR / f"improved_completion_example_{i}.png"
    )

### Evaluate improved model

In [ ]:
# Evaluate improved model
improved_completion_metrics = evaluate_completion_model(
    completion_generator,
    edge_generator,
    test_loader,
    max_batches=100,
    use_predicted_edges=True,
)

improved_completion_metrics

In [ ]:
# Save improved model/history/metrics
torch.save(
    completion_generator.state_dict(),
    MODEL_DIR / "completion_generator_edge_guided_improved.pt"
)

with open(HISTORY_DIR / "completion_training_history_improved.json", "w") as f:
    json.dump({
        "improved_train_losses": improved_train_losses,
        "improved_val_losses": improved_val_losses,
    }, f, indent=2)

with open(METRIC_DIR / "completion_metrics_improved.json", "w") as f:
    json.dump(improved_completion_metrics, f, indent=2)

print("Saved improved completion model.")
print("Improved metrics:", improved_completion_metrics)